# Phase 2 fine-tuning: DINOv2-S retrieval backbone (Colab GPU run)

Stage 2 of the fine-tuning pipeline (`CLAUDE.md` → "Phase 2 Fine-Tuning" Investigation Log). Stage 1
(data prep / training script / eval harness) was already built and verified locally; the only piece
that needs a real GPU is this training run itself. Everything else (manifest generation, ONNX export,
recall evaluation) is CPU-only and already runs fine on the dev machine — don't repeat those here.

**Before running this notebook**: upload `finetune_data_dev_validation.zip` (the dev+validation
flights' satellite maps + drone images, ~3.8GB) to your Google Drive. Any folder is fine — just set
`ZIP_PATH` below to match where you put it.

**Runtime → Change runtime type → T4 GPU** (or better) before running, if not already selected.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set this to wherever you uploaded the zip in your Drive.
ZIP_PATH = "/content/drive/MyDrive/finetune_data_dev_validation.zip"

import os
assert os.path.exists(ZIP_PATH), f"Not found: {ZIP_PATH} -- upload the zip to Drive first and fix this path."
print("Found:", ZIP_PATH, f"({os.path.getsize(ZIP_PATH) / 1e9:.2f} GB)")

In [ ]:
# Clone the repo (public) for the scripts + the already-committed manifest/ground-truth CSVs.
!git clone --depth 1 https://github.com/Eladi24/Optical-Drone-Navigation.git
%cd Optical-Drone-Navigation

In [ ]:
# Unzip the heavy training data (satellite maps + drone images) into place.
# This mirrors the repo-relative layout Images/... and Datasets/... exactly,
# so nothing in the scripts needs path changes.
!unzip -q "{ZIP_PATH}" -d .
!ls Images/ | grep map_clean_uavvisloc
!du -sh Datasets/UAV_VisLoc_dataset/*/drone 2>/dev/null

In [ ]:
# Confirm the manifest came through via git (small, already committed -- see
# scripts/prepare_finetune_dataset.py; no need to regenerate it here).
!wc -l "CSV Files/finetune_manifest.csv"

import torch
print("torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected -- check Runtime > Change runtime type."

In [ ]:
# Colab ships a matched torch/torchvision pair already -- do NOT reinstall
# either from a different index (this project hit a real torch/torchvision
# ABI mismatch doing exactly that in an earlier session, see CLAUDE.md's
# ONNX Runtime build notes). Only add the packages Colab doesn't ship.
!pip install -q timm onnx onnxruntime onnxscript

In [ ]:
# The actual GPU-only step. ~30-60 min on a T4 for 15 epochs / 3500 pairs
# per this project's own estimate -- adjust --epochs down first if you just
# want to sanity-check the loss curve before committing to a full run.
!python scripts/finetune_retrieval_backbone.py --device cuda \
    --epochs 15 --batch-size 16 \
    --checkpoint checkpoints/dinov2_s_finetuned.pt

In [ ]:
# Save the checkpoint back to Drive so it survives the Colab session ending.
!mkdir -p /content/drive/MyDrive/dinov2_finetune_output
!cp checkpoints/dinov2_s_finetuned.pt /content/drive/MyDrive/dinov2_finetune_output/
print("Saved to /content/drive/MyDrive/dinov2_finetune_output/dinov2_s_finetuned.pt")
print("Also downloading directly to your browser as a fallback:")
from google.colab import files
files.download("checkpoints/dinov2_s_finetuned.pt")

## Next steps (back on the dev machine, not here)

Copy `dinov2_s_finetuned.pt` into the repo's `checkpoints/` directory, then run these two CPU-only
steps locally (no GPU needed, this environment already has everything else on disk):

```bash
source ../CV_IP/cv_env/bin/activate

# Export to ONNX and validate PyTorch vs. ONNX Runtime agree
python3 scripts/export_retrieval_backbone.py --backbone dinov2_s \
    --weights checkpoints/dinov2_s_finetuned.pt \
    --output models/dinov2_s_finetuned_retrieval.onnx --validate

# Compare recall@k against the frozen baseline (6.9% pooled recall@1 on
# dev+validation, ~7.5% recall@1 across all 9 flights -- see CLAUDE.md)
python3 scripts/eval_retrieval_recall.py --weights models/dinov2_s_finetuned_retrieval.onnx
```

That gives a real fine-tuned-vs-frozen comparison, including the held-out flights (02/05/06/11) this
training run never touched — a genuine zero-shot generalization check, not just a training-set fit.